# Credit Agreement Analyzer - Phase 4 Validation
Testing Q&A engine against the real Ribbon Communications credit agreement.

Prerequisites: Ollama running locally with `llama3:8b` loaded.

In [1]:
import contextlib
import time
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker, build_search_text
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))

c:\Users\kbott\Projects\credit-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Rebuild Pipeline (Phase 1-3)
Re-run extraction through retrieval setup.

In [2]:
# Phase 1: Extract, detect, parse, chunk
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)


defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
preamble_sections = [s for s in sections if s.section_type == "preamble"]
preamble_text = preamble_sections[0].text if preamble_sections else None
print(f"Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

Pages: 289 | Sections: 157 | Terms: 391 | Chunks: 705


In [3]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()}")
print(f"Available: {llm.is_available()}")

Provider: claude-sonnet-4-20250514
Available: True


In [4]:
# Phase 3: Retrieval layer
embedder = Embedder()

start = time.time()
embeddings = embedder.embed([build_search_text(c) for c in chunks])
print(f"Embedded {len(chunks)} chunks in {time.time() - start:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_p4")
store.create_collection("ribbon_p4")
store.add_chunks("ribbon_p4", chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)
print("Retrieval layer ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1388.42it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 705 chunks in 23.8s
Retrieval layer ready


## Step 1: Basic Q&A
Test single questions with no conversation history.

In [5]:
qa = QAEngine(retriever=retriever, llm=llm)
if preamble_text:
    qa.set_preamble(preamble_text)
print("QAEngine initialized")

QAEngine initialized


In [6]:
def print_response(resp: QAResponse) -> None:
    """Display a QAResponse in a readable format."""
    print(f"Answer:\n{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  Section {s.section_id} - {s.section_title} (pp. {pages})")
        if s.relevant_text_snippet:
            print(f"    Snippet: {s.relevant_text_snippet[:120]}...")
    print(f"\nChunks retrieved: {len(resp.retrieved_chunks)}")
    print(f"LLM time: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used}")
    print("-" * 80)

In [7]:
# Q1: Total revolving commitment (exact dollar amount)
qa.clear_history()
print("Q: What is the total revolving commitment amount?\n")
resp = qa.ask("What is the total revolving commitment amount?", "ribbon_p4")
print_response(resp)

Q: What is the total revolving commitment amount?

Answer:
The total revolving commitment amount is $35,000,000.

This is stated in multiple places in the credit agreement:

1. In the recitals, which specify that the Lenders have agreed to extend credit facilities "consisting of a term loan facility in the aggregate principal amount of $350,000,000, and a revolving loan facility to the Borrower in an aggregate principal amount of $35,000,000, including a letter of credit sub-facility"

2. In the definition of "Total Revolving Commitments," which states "The original amount of the Total Revolving Commitments is $35,000,000"

3. In the definition of "Revolving Commitment," which also confirms "The original amount of the Total Revolving Commitments is $35,000,000"

The revolving facility includes a letter of credit sub-facility, with the L/C Commitment being a sublimit of the Total Revolving Commitments rather than additional capacity.

Confidence: HIGH
Sources (1):
  Section 1.1 - Define

In [8]:
# Q2: Financinal covenants (list them)
qa.clear_history()
print("Q: What financial covenants are included in the agreement?\n")
resp = qa.ask("What financial covenants are included in the agreement?", "ribbon_p4")
print_response(resp)

Q: What financial covenants are included in the agreement?

Answer:
Based on the provided context, the credit agreement contains one financial covenant:

Maximum Consolidated Net Leverage Ratio (Section 7.1(a))

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following thresholds, tested quarterly starting September 30, 2024:

- September 30, 2024 through December 31, 2025: 4.75:1.00
- March 31, 2026 and thereafter: 4.00:1.00

The Consolidated Net Leverage Ratio is calculated as:
(Consolidated Funded Indebtedness minus Unrestricted Cash up to $50 million) divided by Consolidated Adjusted EBITDA for the trailing four quarters.

Equity Cure Provision (Section 7.1(b))

If the borrower fails the leverage test, it can cure the breach by making a cash equity contribution within 10 business days after financial statements are due. Key mechanics:

1. The equity contribution gets added to Consolidated Adjusted EBITDA for covenant calculation purposes
2. Dur

In [9]:
# Q3: Incremental debt capacity (your core use case)
qa.clear_history()
print("Q: How much incremental debt can the borrower incur?\n")
resp = qa.ask("How much incremental debt can the borrower incur?", "ribbon_p4")
print_response(resp)

Q: How much incremental debt can the borrower incur?

Answer:
The borrower can incur incremental debt up to the "Available Incremental Amount," which is defined as the greater of:

1. $100,000,000 (fixed dollar basket), or
2. Ratio Incremental Amount (leverage-based basket)

The Ratio Incremental Amount allows incremental debt based on pro forma leverage ratios, with different caps depending on the security structure:

- Pari passu secured debt: Up to 2.90x Consolidated First Lien Net Leverage Ratio
- Junior lien secured debt: Up to 3.40x Consolidated Secured Net Leverage Ratio  
- Unsecured debt: Up to 3.90x Consolidated Net Leverage Ratio

Key Constraints:

1. Revolving/Term Mix: After any revolving commitment increase, the ratio of total revolving commitments to term loans cannot exceed 10%

2. Pricing Step-Up: For pari passu secured incremental facilities, if the all-in yield exceeds the existing facilities by more than 0.50% per annum, the existing facilities' pricing must be incr

In [10]:
# Q4: Restricted payments (complex multi-basket answer)
qa.clear_history()
print("Q: What restricted payments are permitted under the credit agreement?\n")
resp = qa.ask("What restricted payments are permitted under the credit agreement?", "ribbon_p4")
print_response(resp)

Q: What restricted payments are permitted under the credit agreement?

Answer:
Based on Section 7.6, the credit agreement permits several categories of restricted payments, provided no Event of Default has occurred and is continuing or would result from the payment:

SUBSIDIARY DISTRIBUTIONS
1. Any Subsidiary may make restricted payments to any Loan Party
2. Non-Loan Party subsidiaries may make restricted payments to other Group Members or to Capital Stock holders on a pro rata basis (or more favorable to Loan Parties)

EMPLOYEE-RELATED PAYMENTS
3. Purchase of Capital Stock or stock options from current/former directors, officers, or employees with tiered limits:
   - $2.5 million annually (unused amounts carry forward)
   - Plus $5 million if Consolidated Net Leverage Ratio ≤ 2.40:1.00 after the payment
   - Plus $10 million if Consolidated Net Leverage Ratio ≤ 1.90:1.00 after the payment
4. Stock dividends payable solely in common stock
5. Payments for employee taxes related to retir

In [11]:
# Q5: NOT FOUND test - question about something unlikely to be in the doc
qa.clear_history()
print("Q: What are the details of the borrower's pet insurance policy?\n")
resp = qa.ask("What are the details of the borrower's pet insurance policy?", "ribbon_p4")
print_response(resp)

Q: What are the details of the borrower's pet insurance policy?

Answer:
Based on my review of the provided credit agreement context, there is no information about the borrower's pet insurance policy. 

The credit agreement contains a general insurance provision in Section 4.24 that requires the Loan Parties to maintain insurance with financially sound and reputable insurance companies on their property in amounts and against risks that are usually insured against by companies in similar businesses. However, this section does not specify any details about pet insurance or any other specific types of insurance policies.

The insurance provision is focused on standard business insurance requirements rather than specific policy details. To find information about specific insurance policies, including any pet insurance, you would need to check:

1. The insurance schedules or exhibits (if any)
2. The borrower's compliance certificates 
3. Separate insurance documentation outside of this cre

## Step 2: Conversation Follow-ups
Test that question reformulation works for multi-turn Q&A.

In [12]:
qa.clear_history()

# Turn 1: Establish context
print("Turn 1: What is the term loan facility size?\n")
resp1 = qa.ask("What is the term loan facility size?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up using "it" (requires reformulation)
print("\nTurn 2: What is the interest rate on it?\n")
resp2 = qa.ask("What is the interest rate on it?", "ribbon_p4")
print_response(resp2)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What is the term loan facility size?

Answer:
The term loan facility size is $350,000,000.

This is stated in the Recitals section, which indicates that "the Lenders have agreed to extend certain credit facilities to the Borrower...in an aggregate principal amount not to exceed $385,000,000, consisting of a term loan facility in the aggregate principal amount of $350,000,000, and a revolving loan facility to the Borrower in an aggregate principal amount of $35,000,000."

The term loan facility can potentially be increased through the Incremental Facility provisions in Section 2.27, which allows for Incremental Term Facilities up to the Available Incremental Amount, subject to various conditions including leverage ratio tests.

Confidence: HIGH
Sources (1):
  Section 2.27 - Incremental Facility (pp. 83, 84)
    Snippet: 2.27 Incremental Facility.  (a) At any time after the Closing Date during the Commitment Period (in the case of a Revolv...

Chunks retrieved: 15
LLM time: 18.1s

In [13]:
qa.clear_history()

# Turn 1: Ask about covenants broadly
print("Turn 1: What financial covenants does this agreement have?\n")
resp1 = qa.ask("What financial covenants does this agreement have?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up referencing the previous answer
print("\nTurn 2: Are there any step-downs in those levels?\n")
resp2 = qa.ask("Are there any step-downs in those levels?", "ribbon_p4")
print_response(resp2)

# Turn 3: Another follow-up
print("\nTurn 3: What happens if the borrower breaches them?\n")
resp3 = qa.ask("What happens if the borrower breaches them?", "ribbon_p4")
print_response(resp3)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What financial covenants does this agreement have?

Answer:
Based on the provided context, this credit agreement contains one financial covenant:

Maximum Consolidated Net Leverage Ratio Covenant

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following step-down schedule (Section 7.1(a)):

1. September 30, 2024 through December 31, 2025: 4.75:1.00
2. March 31, 2026 and thereafter: 4.00:1.00

The covenant is tested quarterly based on the last day of any period of four consecutive trailing fiscal quarters of Holdings, beginning with the quarter ending September 30, 2024.

Key Details:
- The covenant steps down from 4.75x to 4.00x starting March 31, 2026
- Testing begins September 30, 2024 (no financial covenant testing before then)
- Based on trailing four-quarter periods
- Uses "Consolidated Net Leverage Ratio" as defined in the agreement

The context does not show any other financial covenants such as minimum EBITDA, interest coverage rat

## Step 3: Context Assembly Inspection
Look at the raw prompt being sent to the LLM to verify structure.

In [14]:
from credit_analyzer.generation.prompts import build_context_prompt

# Manually retrieve and build context to inspect
result = retriever.retrieve(
    query="What is the total revolving commitment amount?",
    document_id="ribbon_p4",
    top_k=5,
)

prompt = build_context_prompt(
    chunks=result.chunks,
    definitions=result.injected_definitions,
    history=[],
    question="What is the total revolving commitment amount?",
)

print(f"Prompt length: {len(prompt)} chars")
print(f"Approximate tokens: ~{len(prompt) // 4}")
print("\n" + "=" * 80)
print(prompt)
print("=" * 80)

Prompt length: 8214 chars
Approximate tokens: ~2053

=== CONTEXT FROM CREDIT AGREEMENT ===

--- Source: Defined Terms (Section 1.1, Pages 55) ---
“Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment is a sublimit of the Total Revolving
Commitments.

--- Source: Defined Terms (Section 1.1, Pages 50) ---
“Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth under the heading “Revolving Commitment” opposite such Lender’s name on
Schedule 1.1A or in the Assignment and Assumption, Incremental Joinder, Extension Amendment or Refinancing Amendment, as
applicable, pursuant to which such Lender became a party hereto, as the same may be changed from time to time pursuant to the terms
hereof (including in connection with assignments an

## Step 4: Timing & Token Budget
Verify we're within the 8K context window and check response times.

In [15]:
test_questions = [
    "What is the total revolving commitment amount?",
    "What are the financial covenant test levels?",
    "How much incremental debt can the borrower incur?",
    "Who is the administrative agent?",
    "What are the mandatory prepayment triggers?",
]

print(f"{'Question':<55} {'Time':>6} {'Tokens':>7} {'Conf':>6}")
print("-" * 80)

for q in test_questions:
    qa.clear_history()
    start = time.time()
    resp = qa.ask(q, "ribbon_p4")
    elapsed = time.time() - start
    print(
        f"{q:<55} {elapsed:>5.1f}s {resp.llm_response.tokens_used:>7} {resp.confidence:>6}"
    )

Question                                                  Time  Tokens   Conf
--------------------------------------------------------------------------------
What is the total revolving commitment amount?           33.0s     280   HIGH
What are the financial covenant test levels?             25.2s     437   HIGH


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CYbXi2UeNZKNoWf6WWh5J'}

In [ ]:
# Re-import (assumes retriever, defn_index, bm25 already built)
from collections import Counter

from credit_analyzer.config import (
    DEFINITION_UBIQUITY_THRESHOLD,
    MAX_DEFINITIONS_INJECTED,
    QA_DEFINITION_MAX_CHARS,
    QA_SECTION_TYPES_EXCLUDE,
)

In [ ]:
# === STEP 1: What does the Applicable Margin definition actually look like? ===
am_defn = defn_index.lookup("Applicable Margin")
if am_defn:
    print(f"Definition length: {len(am_defn)} chars")
    print(f"Exceeds QA_DEFINITION_MAX_CHARS ({QA_DEFINITION_MAX_CHARS})? {len(am_defn) > QA_DEFINITION_MAX_CHARS}")
    print(f"\nFirst 300 chars:\n{am_defn[:300]}")
    print(f"\nLast 300 chars:\n{am_defn[-300:]}")
else:
    print("ERROR: 'Applicable Margin' not found in definitions index!")
    # Check for close matches
    close = [t for t in defn_index.definitions if 'margin' in t.lower() or 'applicable' in t.lower()]
    print(f"Close matches: {close}")

In [ ]:
# === STEP 2: Is there a definition chunk for Applicable Margin? ===
print(f"Definition chunk lookup has {len(retriever._definition_chunk_lookup)} entries")
am_chunk = retriever._definition_chunk_lookup.get("Applicable Margin")
if am_chunk:
    print(f"Chunk ID: {am_chunk.chunk_id}")
    print(f"Chunk type: {am_chunk.chunk_type}")
    print(f"Token count: {am_chunk.token_count}")
    print(f"Text length: {len(am_chunk.text)} chars")
    print(f"Defined terms present: {am_chunk.defined_terms_present}")
    print(f"\nFirst 200 chars of chunk text:\n{am_chunk.text[:200]}")
else:
    print("ERROR: No definition chunk for 'Applicable Margin'!")
    # Check what terms DO have chunks
    margin_chunks = {t: c for t, c in retriever._definition_chunk_lookup.items()
                     if 'margin' in t.lower() or 'applicable' in t.lower()}
    print(f"Chunks with 'margin' or 'applicable': {list(margin_chunks.keys())}")

In [ ]:
# === STEP 3: Is Applicable Margin flagged as ubiquitous? ===
am_freq = retriever._term_doc_freq.get("Applicable Margin", 0.0)
print(f"Applicable Margin doc frequency: {am_freq:.3f}")
print(f"Ubiquity threshold: {DEFINITION_UBIQUITY_THRESHOLD}")
print(f"Is ubiquitous: {'Applicable Margin' in retriever._ubiquitous_terms}")
print(f"\nTotal ubiquitous terms: {len(retriever._ubiquitous_terms)}")
print(f"Sample ubiquitous terms: {list(retriever._ubiquitous_terms)[:20]}")

In [ ]:
# === STEP 4: Run retrieval and trace what happens ===
query = "What is the interest rate on the term loan?"

# Run retrieval
result = retriever.retrieve(
    query=query,
    document_id="ribbon_p4",
    top_k=15,
    section_types_exclude=QA_SECTION_TYPES_EXCLUDE,
)

print(f"Retrieved {len(result.chunks)} chunks")
print(f"Injected {len(result.injected_definitions)} definitions")
print()

# Show what was retrieved
for i, hc in enumerate(result.chunks):
    print(f"  [{i}] score={hc.score:.3f} src={hc.source:10s} "
          f"type={hc.chunk.chunk_type:10s} "
          f"sec={hc.chunk.section_id:6s} {hc.chunk.section_title[:50]}")

print(f"\nInjected definitions: {list(result.injected_definitions.keys())}")
for term, defn in result.injected_definitions.items():
    print(f"  {term}: {len(defn)} chars (truncated={len(defn) > QA_DEFINITION_MAX_CHARS})")

In [ ]:
# === STEP 5: Replay _inject_and_expand_definitions manually ===
# This is the critical trace - shows exactly why promotion fails/succeeds

# Get the pre-injection chunks (re-run retrieval without injection)
result_raw = retriever.retrieve(
    query=query,
    document_id="ribbon_p4",
    top_k=15,
    section_types_exclude=QA_SECTION_TYPES_EXCLUDE,
    inject_definitions=False,
)

# Pass 1: Find terms in retrieved chunks
query_terms = set(defn_index.find_terms_in_text(query))
print(f"Terms detected in query: {query_terms}")
print()

chunk_term_counts: Counter[str] = Counter()
for hc in result_raw.chunks:
    terms = defn_index.find_terms_in_text(hc.chunk.text)
    chunk_term_counts.update(terms)
    # Show which chunks mention Applicable Margin
    if "Applicable Margin" in terms:
        print(f"  Chunk mentions 'Applicable Margin': {hc.chunk.section_id} - {hc.chunk.section_title[:40]}")

print(f"\nTotal unique terms found across chunks: {len(chunk_term_counts)}")
print(f"'Applicable Margin' count in chunks: {chunk_term_counts.get('Applicable Margin', 0)}")

In [ ]:
# === STEP 6: Term scoring and ranking ===
all_candidate_terms = set(chunk_term_counts.keys()) | query_terms
print(f"Total candidate terms: {len(all_candidate_terms)}")

term_scores: dict[str, float] = {}
for term in all_candidate_terms:
    score = float(chunk_term_counts.get(term, 0))
    if term in query_terms:
        score += 100.0
    elif term in retriever._ubiquitous_terms:
        score -= 50.0
    term_scores[term] = score

ranked = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)

print("\nTop 25 terms by score:")
for term, score in ranked[:25]:
    ubiq = " [UBIQUITOUS]" if term in retriever._ubiquitous_terms else ""
    in_q = " [IN_QUERY]" if term in query_terms else ""
    has_chunk = " [HAS_CHUNK]" if term in retriever._definition_chunk_lookup else ""
    defn = defn_index.lookup(term)
    defn_len = len(defn) if defn else 0
    is_long = " [LONG]" if defn_len > QA_DEFINITION_MAX_CHARS else ""
    print(f"  {score:7.1f}  {term[:45]:45s}  defn={defn_len:5d} chars{ubiq}{in_q}{has_chunk}{is_long}")

# Where does Applicable Margin rank?
am_rank = next((i for i, (t, _) in enumerate(ranked) if t == "Applicable Margin"), -1)
am_score = term_scores.get("Applicable Margin", -999)
print(f"\n'Applicable Margin' rank: {am_rank} (score: {am_score})")
print(f"MAX_DEFINITIONS_INJECTED: {MAX_DEFINITIONS_INJECTED}")
print(f"Would be in primary_terms? {am_rank < MAX_DEFINITIONS_INJECTED if am_rank >= 0 else 'NOT FOUND'}")

In [ ]:
# === STEP 7: Promotion gate check for Applicable Margin ===
am_defn = defn_index.lookup("Applicable Margin")
am_chunk = retriever._definition_chunk_lookup.get("Applicable Margin")

if am_defn:
    is_long = len(am_defn) > QA_DEFINITION_MAX_CHARS
    is_ubiq = "Applicable Margin" in retriever._ubiquitous_terms
    in_query = "Applicable Margin" in query_terms
    has_chunk = am_chunk is not None

    print("Promotion gate for 'Applicable Margin':")
    print(f"  is_long ({len(am_defn)} > {QA_DEFINITION_MAX_CHARS}): {is_long}")
    print(f"  has_chunk: {has_chunk}")
    print(f"  is_ubiquitous: {is_ubiq}")
    print(f"  in_query: {in_query}")
    print("  Promotion condition (is_long AND has_chunk AND (NOT ubiq OR in_query)):")
    would_promote = is_long and has_chunk and (not is_ubiq or in_query)
    print(f"  -> {would_promote}")

    if not would_promote:
        print("\n  WHY NOT PROMOTED:")
        if not is_long:
            print(f"    Definition is only {len(am_defn)} chars, under the {QA_DEFINITION_MAX_CHARS} threshold.")
            print(f"    It will be TRUNCATED to {QA_DEFINITION_MAX_CHARS} chars and injected as text.")
            print("    This is the bug: the pricing grid gets cut off.")
        if not has_chunk:
            print("    No definition chunk exists for this term.")
        if is_ubiq and not in_query:
            print(f"    Flagged ubiquitous (freq={am_freq:.3f}) and not in query.")
else:
    print("Applicable Margin not in definitions index!")

In [ ]:
# === STEP 8: What actually gets injected for Applicable Margin? ===
if "Applicable Margin" in result.injected_definitions:
    injected_text = result.injected_definitions["Applicable Margin"]
    print(f"Injected text length: {len(injected_text)} chars")
    print(f"Full injected text:\n{injected_text}")
else:
    print("'Applicable Margin' was NOT injected as a definition.")
    # Check if it was promoted as a chunk
    promoted = [hc for hc in result.chunks if hc.source == 'definition']
    if promoted:
        print(f"Promoted definition chunks: {[(hc.chunk.defined_terms_present, hc.chunk.chunk_id) for hc in promoted]}")
    else:
        print("No definition chunks were promoted either.")
        print("Applicable Margin is COMPLETELY MISSING from context.")

In [ ]:
# === STEP 9: Check if the definition chunk text vs definition text match ===
# The chunk text and the definitions index text may diverge
if am_chunk and am_defn:
    print(f"Definition index text length: {len(am_defn)} chars")
    print(f"Definition chunk text length: {len(am_chunk.text)} chars")
    print(f"Are they identical? {am_chunk.text == am_defn}")
    if am_chunk.text != am_defn:
        # Show where they diverge
        for i, (a, b) in enumerate(zip(am_chunk.text, am_defn)):
            if a != b:
                print(f"  First difference at char {i}")
                print(f"  Chunk: ...{am_chunk.text[max(0,i-20):i+20]}...")
                print(f"  Defn:  ...{am_defn[max(0,i-20):i+20]}...")
                break
        print(f"  Chunk text has {am_chunk.text.count(chr(10))} newlines")
        print(f"  Defn text has {am_defn.count(chr(10))} newlines")

In [ ]:
# === STEP 10: Other important long definitions that might fail ===
print("Definitions over QA_DEFINITION_MAX_CHARS that would need promotion:\n")
long_defs = []
for term, defn in defn_index.definitions.items():
    if len(defn) > QA_DEFINITION_MAX_CHARS:
        has_chunk = term in retriever._definition_chunk_lookup
        is_ubiq = term in retriever._ubiquitous_terms
        long_defs.append((term, len(defn), has_chunk, is_ubiq))

long_defs.sort(key=lambda x: x[1], reverse=True)
print(f"{'Term':<45s} {'Chars':>6s} {'HasChunk':>8s} {'Ubiq':>5s}")
print("-" * 70)
for term, chars, has_chunk, is_ubiq in long_defs[:30]:
    print(f"{term[:45]:<45s} {chars:>6d} {str(has_chunk):>8s} {str(is_ubiq):>5s}")

In [ ]:
from credit_analyzer.config import QA_SECTION_TYPES_EXCLUDE

# What chunks exist for Section 7.1?
sec71_chunks = [c for c in chunks if c.section_id == '7.1']
print(f"Section 7.1 chunks: {len(sec71_chunks)}\n")
for c in sec71_chunks:
    print(f"  {c.chunk_id} | type={c.chunk_type} | tokens={c.token_count} | pp {c.page_numbers}")
    print(f"  First 200 chars: {c.text[:200]}")
    print()

In [ ]:
# Compare retrieval for the two phrasings
q1 = "What financial covenants are included in the agreement?"
q2 = "What financial covenants does this agreement have?"

r1 = retriever.retrieve(q1, "ribbon_p4", top_k=15,
                        section_types_exclude=QA_SECTION_TYPES_EXCLUDE,
                        inject_definitions=False)
r2 = retriever.retrieve(q2, "ribbon_p4", top_k=15,
                        section_types_exclude=QA_SECTION_TYPES_EXCLUDE,
                        inject_definitions=False)

print("=== Q1: 'included in the agreement' ===")
for i, hc in enumerate(r1.chunks):
    is71 = ' <<<' if hc.chunk.section_id == '7.1' else ''
    has_ratio = ' [HAS 4.75]' if '4.75' in hc.chunk.text else ''
    print(f"  [{i:2d}] {hc.score:.3f} {hc.source:6s} {hc.chunk.section_id:6s} {hc.chunk.section_title[:40]}{is71}{has_ratio}")

print(f"\n=== Q2: 'does this agreement have' ===")
for i, hc in enumerate(r2.chunks):
    is71 = ' <<<' if hc.chunk.section_id == '7.1' else ''
    has_ratio = ' [HAS 4.75]' if '4.75' in hc.chunk.text else ''
    print(f"  [{i:2d}] {hc.score:.3f} {hc.source:6s} {hc.chunk.section_id:6s} {hc.chunk.section_title[:40]}{is71}{has_ratio}")

# Which 7.1 chunks made it?
r1_71 = {hc.chunk.chunk_id for hc in r1.chunks if hc.chunk.section_id == '7.1'}
r2_71 = {hc.chunk.chunk_id for hc in r2.chunks if hc.chunk.section_id == '7.1'}
print(f"\nQ1 has 7.1 chunks: {r1_71}")
print(f"Q2 has 7.1 chunks: {r2_71}")
print(f"In Q1 but not Q2: {r1_71 - r2_71}")
print(f"In Q2 but not Q1: {r2_71 - r1_71}")

In [ ]:
# But wait - the multi-turn case reformulates the query.
# Q2 in the notebook is Turn 1, so no reformulation.
# Check: is 7.1(a) a single chunk or split across chunks?
for c in sec71_chunks:
    if '4.75' in c.text or '4.00' in c.text:
        print(f"Chunk with covenant thresholds: {c.chunk_id}")
        print(f"  tokens={c.token_count}, chars={len(c.text)}")
        print(f"  Text:\n{c.text[:500]}")
        print()

In [ ]:
store.delete_collection("ribbon_p4")
print("Cleaned up test collection")